In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [2]:

import os
import pandas as pd
import polars as pl
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import kaggle_evaluation.default_inference_server

global_model = None

def predict(test: pl.DataFrame) -> float:
    global global_model
    
    if global_model is None:
        print("First call detected. Training model now...")
        
        train_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
        
        features = ['M1', 'P1', 'E1', 'I1'] 
        target = 'market_forward_excess_returns'
        
        train_df = train_df.dropna(subset=features + [target])

        model = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42, n_jobs=-1)
        model.fit(train_df[features], train_df[target])
        
        global_model = model
        print("Training complete!")
    
    features = ['M1', 'P1', 'E1', 'I1'] 
    
    test_pd = test.to_pandas()
    
    X_test = pd.DataFrame(index=test_pd.index)
    for f in features:
        if f in test_pd.columns:
            X_test[f] = test_pd[f]
        else:
            X_test[f] = 0.0
            
    X_test = X_test.fillna(0)
    
    raw_prediction = global_model.predict(X_test[features])[0]
    
    if raw_prediction > 0:
        allocation = 1.0
    else:
        allocation = 0.0
        
    return float(max(0.0, min(2.0, allocation)))


inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    print("Running local gateway for testing...")
    inference_server.run_local_gateway(
        ('/kaggle/input/hull-tactical-market-prediction/',) 
    )

Running local gateway for testing...
First call detected. Training model now...
Training complete!
